# 03. Magnitude MP-PCA Denoising — Open QSM Dataset

`02_Add_noise_Open_QSM_FULL_VOLUME_REVISED.ipynb`에서 생성한
**5-echo noisy complex GRE** 데이터를 magnitude로 변환한 뒤,
DIPY의 MP-PCA를 적용하여 denoising을 수행한다.

이번 실험 원칙:

- 입력: `(X, Y, Z, 5)` noisy complex GRE
- Noise level: 10%, 30%, 50%
- Magnitude 변환 후 MP-PCA 수행
- **Brain mask는 denoising에 사용하지 않음**
- MP-PCA는 전체 volume에 적용
- Brain mask는 이후 최종 성능평가 ROI용으로만 보관
- Patch radius = 3 → `7 × 7 × 7` spatial patch


In [ ]:
import numpy as np
import scipy.io as sio
from pathlib import Path
import matplotlib.pyplot as plt
from dipy.denoise.localpca import mppca


### 환경설정

새로운 Open QSM noise 데이터는 다음 폴더에서 읽는다.

`./data/noisy_data/`

파일명:

- `qsm_siemens_monopolar_noisy_10.npz`
- `qsm_siemens_monopolar_noisy_30.npz`
- `qsm_siemens_monopolar_noisy_50.npz`

Magnitude MP-PCA는 모든 noise level에서 동일한 `PATCH_R=3`을 사용한다.


In [ ]:
NOISE_LEVELS = [10, 30, 50]

# 기존 졸업논문 실험 설정
PATCH_R = 3  # 7 x 7 x 7 spatial patch

NOISY_DIR = Path("./data/noisy_data")
OUT_DIR = Path("./data/denoised/magnitude")

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Noisy directory :", NOISY_DIR.resolve())
print("Output directory:", OUT_DIR.resolve())
print("Patch radius    :", PATCH_R)
print(
    "Patch size      :",
    f"{2 * PATCH_R + 1} x {2 * PATCH_R + 1} x {2 * PATCH_R + 1}"
)


### 1. 새 QSM Noise 데이터 불러오기

각 noise level의 `.npz` 파일에서 다음 정보를 불러온다.

- `noisy_complex`: `(X, Y, Z, 5)` complex GRE
- `mask_brain`: 평가용 brain mask
- `noise_ratio`
- `sigma`
- `seed`
- `overall_mean`
- `echo_times`
- `field_strength`

`mask_brain`은 **불러와서 결과 파일에 함께 저장만 하며, MP-PCA에는 사용하지 않는다.**


In [ ]:
noisy_datasets = {}

for noise_level in NOISE_LEVELS:
    noisy_path = (
        NOISY_DIR /
        f"qsm_siemens_monopolar_noisy_{noise_level}.npz"
    )

    if not noisy_path.exists():
        raise FileNotFoundError(
            f"Noise 파일을 찾을 수 없습니다: {noisy_path}\n"
            "먼저 02_Add_noise_Open_QSM_FULL_VOLUME_REVISED.ipynb를 실행하세요."
        )

    data = np.load(noisy_path, allow_pickle=False)

    noisy_complex = data["noisy_complex"].astype(np.complex64)
    mask = data["mask_brain"].astype(bool)

    if noisy_complex.ndim != 4:
        raise ValueError(
            f"{noise_level}% noisy_complex는 4D여야 합니다. "
            f"현재 shape={noisy_complex.shape}"
        )

    if noisy_complex.shape[-1] != 5:
        raise ValueError(
            f"{noise_level}% 데이터는 5 echoes를 예상합니다. "
            f"실제={noisy_complex.shape[-1]}"
        )

    if mask.shape != noisy_complex.shape[:3]:
        raise ValueError(
            f"{noise_level}% mask shape {mask.shape} != "
            f"spatial shape {noisy_complex.shape[:3]}"
        )

    noisy_datasets[noise_level] = {
        "noisy_complex": noisy_complex,
        "mask_brain": mask,
        "noise_ratio": float(data["noise_ratio"]),
        "sigma": float(data["sigma"]),
        "seed": int(data["seed"]),
        "overall_mean": float(data["overall_mean"]),
        "echo_times": (
            data["echo_times"].astype(np.float64)
            if "echo_times" in data.files
            else np.full(5, np.nan, dtype=np.float64)
        ),
        "field_strength": (
            float(data["field_strength"])
            if "field_strength" in data.files
            else np.nan
        ),
    }

    print(
        f"{noise_level}% 불러오기 완료 | "
        f"shape={noisy_complex.shape} | "
        f"dtype={noisy_complex.dtype} | "
        f"sigma={float(data['sigma']):.6f}"
    )


### 2. Complex GRE → Magnitude 변환

Magnitude MP-PCA는 complex 신호 자체가 아니라 각 voxel/echo의 magnitude를 입력으로 사용한다.

`Magnitude = abs(Complex)`

따라서 각 noise level의 `(X,Y,Z,5)` complex GRE를 동일한 shape의 float32 magnitude 배열로 변환한다.


In [ ]:
magnitude_data = {}

for noise_level in NOISE_LEVELS:
    noisy_complex = noisy_datasets[noise_level]["noisy_complex"]

    noisy_mag = np.abs(noisy_complex).astype(np.float32)

    magnitude_data[noise_level] = noisy_mag

    print(
        f"{noise_level}% magnitude 변환 완료 | "
        f"shape={noisy_mag.shape} | "
        f"dtype={noisy_mag.dtype}"
    )


### 3. Magnitude MP-PCA Denoising

DIPY `mppca`를 이용한다.

이번 실험에서는 **brain mask를 MP-PCA에 전달하지 않는다.**
따라서 `(X,Y,Z,5)` magnitude 전체 volume을 대상으로 denoising한다.

Patch 설정:

- `PATCH_R = 3`
- spatial patch size = `7 × 7 × 7`
- 한 patch의 spatial voxel 수 = `343`
- 각 voxel의 feature 수 = `5 echoes`

즉 local patch에서 대략 `343 × 5` 형태의 magnitude data를 기반으로 MP-PCA가 수행된다.


In [ ]:
denoised_data = {}

for noise_level in NOISE_LEVELS:
    noisy_mag = magnitude_data[noise_level]

    print("\n" + "=" * 60)
    print(f"Noise level   : {noise_level}%")
    print(f"Input shape   : {noisy_mag.shape}")
    print(f"Echoes        : {noisy_mag.shape[-1]}")
    print(f"Patch radius  : {PATCH_R}")
    print(
        "Patch size    : "
        f"{2 * PATCH_R + 1} x "
        f"{2 * PATCH_R + 1} x "
        f"{2 * PATCH_R + 1}"
    )
    print("Brain mask    : NOT USED")
    print("Denoise ROI   : FULL VOLUME")
    print("=" * 60)

    # --------------------------------------------------------
    # 핵심:
    # mask를 전달하지 않고 전체 volume에 Magnitude MP-PCA 수행
    # --------------------------------------------------------
    deno_mag = mppca(
        noisy_mag,
        patch_radius=PATCH_R
    ).astype(np.float32)

    if deno_mag.shape != noisy_mag.shape:
        raise ValueError(
            f"Denoised shape {deno_mag.shape} != input shape {noisy_mag.shape}"
        )

    denoised_data[noise_level] = deno_mag

    print(f"{noise_level}% Magnitude MP-PCA 완료")


### 4. 결과 저장

각 noise level의 Magnitude MP-PCA 결과를 저장한다.

주요 저장 항목:

- `magnitude_denoised`: `(X,Y,Z,5)` denoised magnitude
- `mask_brain`: 이후 **평가 ROI용**
- `noise_level`
- `noise_ratio`
- `sigma`
- `patch_radius`
- `echo_times`
- `field_strength`
- `denoise_region = "full_volume"`

기존 후속 분석 코드와의 호환성을 위해 `.mat` 형식으로 저장한다.


In [ ]:
for noise_level in NOISE_LEVELS:
    meta = noisy_datasets[noise_level]
    deno_mag = denoised_data[noise_level]

    output_path = (
        OUT_DIR /
        f"denoised_magnitude_{noise_level}_r{PATCH_R}.mat"
    )

    save_data = {
        "magnitude_denoised": deno_mag,
        "mask_brain": meta["mask_brain"].astype(np.uint8),
        "noise_level": np.int32(noise_level),
        "noise_ratio": np.float32(meta["noise_ratio"]),
        "sigma": np.float32(meta["sigma"]),
        "seed": np.int32(meta["seed"]),
        "overall_mean": np.float32(meta["overall_mean"]),
        "patch_radius": np.int32(PATCH_R),
        "echo_times": np.asarray(meta["echo_times"], dtype=np.float64),
        "field_strength": np.float32(meta["field_strength"]),
        "denoise_region": "full_volume",
        "mask_usage": "evaluation_only",
    }

    sio.savemat(
        output_path,
        save_data,
        do_compression=True
    )

    print("저장 완료 ->", output_path)


### 5. Denoising 결과 확인

Clean reference 없이도 서버에서 바로 확인할 수 있도록 다음 3가지를 비교한다.

- Noisy Magnitude
- Denoised Magnitude
- Removed component = `Noisy - Denoised`

여기서 Removed component는 **최종 error map이 아니다.**
단순히 MP-PCA가 입력 영상에서 제거한 성분을 확인하기 위한 시각화이다.

최종 PSNR / SSIM / NRMSE / HFEN 평가는 별도 평가 노트북에서 clean reference와 brain mask를 사용한다.


In [ ]:
echo_idx = 0

# 데이터의 중앙 axial slice 사용
sample_shape = magnitude_data[NOISE_LEVELS[0]].shape
slice_idx = sample_shape[2] // 2

for noise_level in NOISE_LEVELS:
    noisy_mag = magnitude_data[noise_level]
    deno_mag = denoised_data[noise_level]

    noisy_2d = noisy_mag[:, :, slice_idx, echo_idx]
    deno_2d = deno_mag[:, :, slice_idx, echo_idx]

    removed_2d = noisy_2d - deno_2d

    # mask 없이 전체 slice 기준으로 display range 설정
    vmin, vmax = np.percentile(noisy_2d, (1, 99))

    dv = float(np.percentile(np.abs(removed_2d), 99))
    dv = max(dv, 1e-8)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))

    ax[0].imshow(
        noisy_2d,
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )
    ax[0].set_title(f"Noisy ({noise_level}%)")

    ax[1].imshow(
        deno_2d,
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )
    ax[1].set_title("Magnitude MP-PCA")

    ax[2].imshow(
        removed_2d,
        cmap="bwr",
        vmin=-dv,
        vmax=dv
    )
    ax[2].set_title("Removed (Noisy - Denoised)")

    for a in ax:
        a.axis("off")

    fig.suptitle(
        f"Full-volume Magnitude MP-PCA | "
        f"Noise {noise_level}% | Echo {echo_idx + 1} | Slice {slice_idx}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()
